Business Problem:

Customer Success Manager asks:

Why is customer 1001 likely to churn?
What complaints have similar customers reported?
What action should we take to retain them?

For the above they would need below tools for Agent
- SQL queries
- Customer notes
- ML model
- Policy documents

Final Architecture:

User Question
      ↓
 Churn Agent
      ↓
 Tool Selection
      ↓
+----------------------+
| SQL Tool             |
| Vector Search Tool   |
| Prediction Tool      |
| Recommendation Tool  |
+----------------------+
      ↓
Retrieved Results
      ↓
LLM
      ↓
Final Answer

In [0]:
#Install Vector Search client
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import requests
import re

w = WorkspaceClient()
vsc = VectorSearchClient(disable_notice=True)

VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"
INDEX_NAME = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"
SOURCE_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"

EMBEDDING_MODEL = "databricks-gte-large-en"
LLM_MODEL = "databricks-meta-llama-3-1-8b-instruct"

GOLD_TABLE = "dbw_agentic_ai_dev.telco_ai.gold_telco"
NOTES_TABLE = "dbw_agentic_ai_dev.telco_ai.customer_notes"

In [0]:
#check endpoint exists or not
VECTOR_SEARCH_ENDPOINT_NAME = "telco-vector-search-endpoint"

try:
    endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print("Endpoint already exists")

except Exception:
    print("Creating endpoint...")

    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )

In [0]:
endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
print(endpoint)

In [0]:
#create index
embedding_dimension = len(
    spark.table(SOURCE_TABLE)
         .select("embedding")
         .first()["embedding"]
)
INDEX_NAME = "dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index"
vsc.create_delta_sync_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    source_table_name=SOURCE_TABLE,
    index_name=INDEX_NAME,
    pipeline_type="TRIGGERED",
    primary_key="customer_id",
    embedding_dimension=embedding_dimension,
    embedding_vector_column="embedding",
    columns_to_sync=[
        "customer_id",
        "note"
    ]
)

In [0]:
vsc.delete_index(
    endpoint_name="telco-vector-search-endpoint",
    index_name="dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index_1"
)

In [0]:
%sql
DESCRIBE DETAIL dbw_agentic_ai_dev.telco_ai.customer_note_embeddings_index;

In [0]:
#assign index
index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
    index_name=INDEX_NAME
)

In [0]:
def call_llm(prompt, max_tokens=300):
    response = w.serving_endpoints.query(
        name=LLM_MODEL,
        messages=[
            ChatMessage(
                role=ChatMessageRole.USER,
                content=prompt
            )
        ],
        max_tokens=max_tokens,
        temperature=0.0
    )

    return response.choices[0].message.content

In [0]:
def extract_customer_id(question):

    match = re.search(
        r"customer\s+([A-Za-z0-9-]+)",
        question,
        re.IGNORECASE
    )

    customer_id = match.group(1).strip() if match else None

    return customer_id

###### Tool 1: SQL Analytics Tool

In [0]:
#LLM selects from predefined SQL actions and then Python maps the selected action to approved SQL.

def choose_sql_action(question):
    prompt = f"""
You are a SQL routing agent.

Choose the best SQL action for the user question.

Available actions:
1. count_churned_customers
2. churn_rate
3. count_month_to_month_customers

Question:
{question}

Return only one action name.
"""

    return call_llm(prompt, max_tokens=20).strip().lower()    


In [0]:
def run_sql_action(action):
    sql_map = {
        "count_churned_customers": """
            SELECT COUNT(*) AS churned_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Churn_Flag = 1
        """,

        "churn_rate": """
            SELECT
                ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 2) AS churn_rate_percent
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
        """,

        "count_month_to_month_customers": """
            SELECT COUNT(*) AS month_to_month_customers
            FROM dbw_agentic_ai_dev.telco_ai.gold_telco
            WHERE Contract = 'Month-to-month'
        """
    }

    if action not in sql_map:
        return f"Unsupported SQL action: {action}"

    return spark.sql(sql_map[action]).collect()

###### Tool 2: Similar Customer Notes Tool

In [0]:
def similar_customer_notes_tool(question, num_results=3):
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL,
        input=[question]
    )

    question_embedding = [
        float(x)
        for x in response.data[0].embedding
    ]

    results = index.similarity_search(
        query_vector=question_embedding,
        columns=["customer_id", "note"],
        num_results=num_results
    )

    rows = results["result"]["data_array"]

    context_lines = []

    for customer_id, note, score in rows:
        context_lines.append(
            f"Customer {int(customer_id)}: {note}"
        )

    context = "\n".join(context_lines)

    return context, rows

###### Tool 3: Customer Notes Tool

In [0]:
def customer_notes_tool(customer_id):

    notes_df = spark.sql(f"""
        SELECT note
        FROM {NOTES_TABLE}
        WHERE customer_id = '{customer_id}'
    """)

    if notes_df.count() == 0:
        return f"No notes found for customer_id = '{customer_id}'"

    notes = notes_df.toPandas()["note"].tolist()

    return "\n".join(notes)

###### Tool 4: Customer Profile Tool

In [0]:
def customer_profile_tool(customer_id):
    customer_df = spark.sql(f"""
        SELECT *
        FROM {GOLD_TABLE}
        WHERE customerID = '{customer_id}'
    """)

    if customer_df.count() == 0:
        return None

    return customer_df.toPandas().iloc[0]

###### Tool 5: Prediction Tool

Workflow:

customer_id
      ↓
query gold_telco
      ↓
build payload
      ↓
call endpoint
      ↓
return prediction

Then Agent can do:

Question:
Will customer 1001 churn?

Agent
 ↓
Prediction Tool
 ↓
Prediction = 1
 ↓
LLM explanation

In [0]:
def prediction_tool(customer_id):
    customer = customer_profile_tool(customer_id)

    if customer is None:
        return {
            "error": f"No customer found for customerID = '{customer_id}'"
        }

    payload = {
        "dataframe_records": [
            {
                "gender": customer["gender"],
                "SeniorCitizen": int(customer["SeniorCitizen"]),
                "Partner": customer["Partner"],
                "Dependents": customer["Dependents"],
                "tenure": int(customer["tenure"]),
                "PhoneService": customer["PhoneService"],
                "MultipleLines": customer["MultipleLines"],
                "InternetService": customer["InternetService"],
                "OnlineSecurity": customer["OnlineSecurity"],
                "OnlineBackup": customer["OnlineBackup"],
                "DeviceProtection": customer["DeviceProtection"],
                "TechSupport": customer["TechSupport"],
                "StreamingTV": customer["StreamingTV"],
                "StreamingMovies": customer["StreamingMovies"],
                "Contract": customer["Contract"],
                "PaperlessBilling": customer["PaperlessBilling"],
                "PaymentMethod": customer["PaymentMethod"],
                "MonthlyCharges": float(customer["MonthlyCharges"]),
                "TotalCharges": float(customer["TotalCharges"]),
                "Tenure_Group": customer["Tenure_Group"]
            }
        ]
    }

    workspace_url = (
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .apiUrl()
        .get()
    )

    endpoint_url = (
        f"{workspace_url}/serving-endpoints/"
        "telco-churn-endpoint/invocations"
    )

    #token = dbutils.secrets.get( scope="databricks-secrets",  key="model-serving-pat" )

    token = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    response = requests.post(
        endpoint_url,
        headers=headers,
        json=payload
    )

    return response.json()

Customer ID
     ↓
customer_profile_tool()
     ↓
Customer Profile
     ↓
prediction_tool()
     ↓
Prediction
     ↓
retention_recommendation_tool()
     ↓
Recommendation

######Tool 6: Retention Recommendation Tool

In [0]:
#create predefined recommendation actions, then let the LLM choose from them.
#LLM chooses recommendation action
#Python maps action → approved recommendation text
#LLM summarizes final answer

RETENTION_ACTIONS = {
    "offer_discount": "Offer a loyalty or retention discount.",
    "offer_support_package": "Provide complimentary technical support or priority support.",
    "service_quality_review": "Escalate the customer to a service quality review team.",
    "billing_review": "Assign a billing specialist to review charges.",
    "no_action": "No immediate retention action is required."
}


In [0]:
def choose_retention_action(profile, prediction, notes):
    prompt = f"""
You are a customer retention agent.

Choose the best retention action from the list.

Available actions:
1. offer_discount
2. offer_support_package
3. service_quality_review
4. billing_review
5. no_action

Customer profile:
Contract: {profile["Contract"]}
MonthlyCharges: {profile["MonthlyCharges"]}
TechSupport: {profile["TechSupport"]}
Tenure: {profile["tenure"]}

Prediction:
{prediction}

Customer notes:
{notes}

Return only one action name.
"""

    return call_llm(prompt, max_tokens=20).strip().lower()

In [0]:
def retention_recommendation_tool(profile, prediction, notes):
    action = choose_retention_action(
        profile=profile,
        prediction=prediction,
        notes=notes
    )

    recommendation = RETENTION_ACTIONS.get(
        action,
        "No valid recommendation action selected."
    )

    return {
        "action": action,
        "recommendation": recommendation
    }

###### Tool Selection

In [0]:
def choose_tool(question):
    tool_prompt = f"""
You are a churn agent. Choose the best tool for the user question.

Available tools:

1. sql_analytics_tool
Use this for counts, totals, churn rate, how many, averages, month-to-month counts, or table summaries.

2. similar_customer_notes_tool
Use this for questions about why, reasons, complaints, dissatisfaction, cancellation, or customer sentiment.

3. prediction_tool
Use this for questions about churn prediction, churn risk, or whether a specific customer may churn.

4. retention_recommendation_tool
Use this for questions about retention actions, customer retention, loyalty, or how to retain a specific customer.

Question:
{question}

Return only one tool name:
sql_analytics_tool
similar_customer_notes_tool
prediction_tool
retention_recommendation_tool
"""

    return call_llm(tool_prompt, max_tokens=20).strip().lower()

###### Churn Agent

In [0]:
def churn_agent(question):
    tool_name = choose_tool(question)
    customer_id = extract_customer_id(question)

    print(f"Tool selected by LLM: {tool_name}")

    if "sql_analytics_tool" in tool_name:
        sql_action = choose_sql_action(question)
        result = run_sql_action(sql_action)

        tool_result = f"""
SQL action: {sql_action}
SQL result: {result}
"""

    elif "similar_customer_notes_tool" in tool_name:
        context, rows = similar_customer_notes_tool(question)

        tool_result = f"""
Retrieved customer notes:
{context}
"""

    elif "prediction_tool" in tool_name:
        if customer_id is None:
            return "Please provide a customer ID for churn prediction."

        prediction = prediction_tool(customer_id)

        tool_result = f"""
Customer ID: {customer_id}
Prediction result: {prediction}
"""

    elif "retention_recommendation_tool" in tool_name:
        if customer_id is None:
            return "Please provide a customer ID for retention recommendation."

        profile = customer_profile_tool(customer_id)

        if profile is None:
            return f"No customer found for customerID = {customer_id}"

        prediction = prediction_tool(customer_id)
        notes = customer_notes_tool(customer_id)

        recommendation = retention_recommendation_tool(
            profile=profile,
            prediction=prediction,
            notes=notes
        )

        tool_result = f"""
Customer ID: {customer_id}

Customer profile:
Contract: {profile["Contract"]}
MonthlyCharges: {profile["MonthlyCharges"]}
TechSupport: {profile["TechSupport"]}
Tenure: {profile["tenure"]}

Prediction:
{prediction}

Customer notes:
{notes}

Recommendation:
{recommendation}
"""

    else:
        return f"Unable to choose a valid tool. LLM returned: {tool_name}"

    final_prompt = f"""
You are a helpful telco churn analyst.

Answer the user's question using only the tool result below.

Question:
{question}

Tool used:
{tool_name}

Tool result:
{tool_result}

Final answer:
"""

    return call_llm(final_prompt)

Question
 ↓
LLM chooses tool
 ↓
Python executes approved tool
 ↓
Tool result
 ↓
LLM final answer

Question
   ↓
choose_tool()
   ↓
SQL Tool
OR
Notes Tool
OR
Prediction Tool
OR
Retention Tool
   ↓
Result
   ↓
LLM
   ↓
Final Answer

In [0]:
print(churn_agent("How many customers churned?"))

In [0]:
print(churn_agent("What is the churn rate?"))

In [0]:
print(churn_agent("How many month-to-month customers are there?"))

In [0]:
print(churn_agent("Why are customers cancelling service?"))

In [0]:
print(churn_agent("Will customer 7590-VHVEG churn?"))

In [0]:
print(churn_agent("How can we retain customer 7590-VHVEG?"))

In [0]:
%sql
select * from dbw_agentic_ai_dev.telco_ai.silver_telco